## Preprocessing

### Table of Contents
- [Dependencies](#dependencies)
- [Load Data & Set Variables](#load-data--set-variables)
- [Define/Categorize Columns](#definecategorize-columns)
- [Fix Inconsistencies](#fix-inconsistencies)
    - [Trim and Lower Case](#trim-and-lower-case-data)
    - [Cast Date Columns to Datetime](#cast-date-columns-to-datetime)
- [Drop Duplicates](#drop-duplicates)
    - [Drop duplicates for data in all columns duplicated](#1-drop-duplicates-for-data-in-all-columns-duplicated)
    - [Drop duplicates per selected columns](#2-drop-duplicates-per-selected-columns)
- [Create Target Column](#create-target-column)
- [Remove and Adjust Records](#remove-and-adjust-records)
    - [Replace Business Type](#replace-business_type)
    - [policy_expiration_date < policy_effective_date](#policy_expiration_date--policy_effective_date)
    - [total_vehicle_count < 0 and prior_year_mileage_000 > 0](#total_vehicle_count--0-and-prior_year_mileage_000--0)
    - [Annual Premium < 0](#annual_premium--0)
- [Add Columns](#add-columns)
  - [Date Columns](#date-columns)
  - ['Vehicle' Columns](#vehicle-columns)
  - [Claim Loss](#claim-loss)
- [Save to parquet](#save-to-parquet)

### Dependencies

In [1]:
import pandas as pd
import numpy as np

from utils import (
    check_duplicates_stats, 
    check_duplicates_per_columns,
)

### Load Data & Set Variables

In [2]:
REAL_DATA = False           # Defines which csv file should be checked/cleaned

train = "../data/orig/train.csv"  # ---- train data ----
real = "../data/orig/score.csv"   # ---- real data ----

In [3]:
if REAL_DATA:
    df = pd.read_csv(real)   # ---- load real data ----
    dataset_type = "real"
else:
    df = pd.read_csv(train) # ---- load train data ----
    dataset_type = "train"

print(f"Dataset type - {dataset_type}:", df.shape)

Dataset type - train: (15075, 31)


### Define/Categorize Columns

In [4]:
ids = ["policy_id", "insured_id"]

not_in_real = ['has_safety_program']

target_columns = [
    "claim_count",
    "target",
    "first_claim_reported_date",
    "claim_paid_amount_current_period",
    "claim_status_current_period",
    "days_to_first_claim_report",
    "total_loss_amount"
]

date_cols = [
    "snapshot_date",
    "policy_effective_date",
    "policy_expiration_date",
    "first_claim_reported_date"
]

target_column = ["target"]

categorical_cols = ['coverage_type', 'business_type', 'state', 'payment_frequency']
cat_cols_direct = ['coverage_type', 'business_type', 'state']
cat_cols_with_unknown = ['payment_frequency']

numeric_cols = [
    'vehicle_count', 'num_heavy_vehicles', 'total_vehicle_count',
    'vehicle_avg_age', 'mileage_per_vehicle',
    'driver_count', 'driver_avg_age',
    'years_in_business',
    'prior_apd_claim_count', 'prior_al_claim_count', 'prior_loss_amount',
    'has_large_prior_loss', 'loss_per_claim',
    'late_payment_count',

    'deductible',
    'coverage_limit_000',
    'annual_premium',
    'risk_score_external', 'risk_score_external_isna',

    'policy_length_days', 'exposure_ratio',
]

## Clean up and Data Preparation

### Fix Inconsistencies
- inconsistent categorical data
    - trim data
    - lower case
- cast date columns to datetime

#### Trim and Lower Case Data

In [5]:
# ---- Trim and Lower Case Data ----

for col in categorical_cols:
    df[col] = df[col].str.strip().str.lower()

print(f'Unique values: {df["business_type"].unique()}')
print(f'Cleaned data: {df.shape}')

Unique values: <ArrowStringArray>
['service', 'trucking', 'construction', 'retail', 'other', 'agriculture']
Length: 6, dtype: str
Cleaned data: (15075, 31)


#### Cast Date Columns to Datetime

In [6]:
# ---- Cast Date Columns to Datetime ----

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

print({df.info()})
print(f'Cleaned data: {df.shape}')

<class 'pandas.DataFrame'>
RangeIndex: 15075 entries, 0 to 15074
Data columns (total 31 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   policy_id                         15075 non-null  str           
 1   insured_id                        15075 non-null  str           
 2   snapshot_date                     15075 non-null  datetime64[us]
 3   policy_effective_date             15075 non-null  datetime64[us]
 4   policy_expiration_date            15075 non-null  datetime64[us]
 5   coverage_type                     15075 non-null  str           
 6   vehicle_count                     15075 non-null  int64         
 7   vehicle_avg_age                   15075 non-null  float64       
 8   driver_count                      15075 non-null  int64         
 9   driver_avg_age                    14472 non-null  float64       
 10  years_in_business                 15075 non-null  float64

### Drop Duplicates

In [7]:
check_cols = ['policy_id']

#### 1. Drop Duplicates for Data in All Columns Duplicated

In [8]:
# ---- Drop duplicates for all columns ----
df = df.drop_duplicates()

print(f'Cleaned data: {df.shape}')

Cleaned data: (15000, 31)


#### 2. Drop Duplicates Per Selected Columns

In [9]:
# ---- Drop duplicates per column ----
df.drop_duplicates(subset=['policy_id'], inplace=True)

print(f'Cleaned data: {df.shape}')

Cleaned data: (15000, 31)


#### Recheck duplicates in dataset

In [10]:
check_duplicates_stats(df, check_cols)

for check_col in check_cols:
    check_duplicates_per_columns(df, check_col)

Data contains duplicates: False
Total duplicate rows: 0 

policy_id: False, index: 0


Total duplicated rows for policy_id: 0


### Create Target Column

In [11]:
if not REAL_DATA:
    df["target"] = (df["claim_count"] > 0).astype(int)

### Remove and Adjust Records
1. update 1 record with 'business_type' == 'agriculture' to 'other'
2. policy_effective_date > policy_expiration_date (10 records)
3. total_vehicle_count == 0 & prior_year_mileage_000 > 0 | driver_count > 0 (2 records)
4. annual_premium < 0 (5 records - assuming those to be cancellations)

#### Replace 'agriculture' with 'other'
- 1 record updated

In [12]:
# ---- Replace 'agriculture' with 'other' ----
# "agriculture" with "other" as all records are from the same target category

df["business_type"] = df["business_type"].replace({"agriculture": "other"})

print(f'Cleaned data: {df.shape}')

Cleaned data: (15000, 32)


#### policy_expiration_date < policy_effective_date
- Drop 10 records

In [13]:
# ---- policy_effective_date > policy_expiration_date ----

df["policy_length_days"] = (df["policy_expiration_date"] - df["policy_effective_date"]).dt.days

In [14]:
df[df["policy_length_days"] < 0]

,policy_id,insured_id,snapshot_date,policy_effective_date,policy_expiration_date,coverage_type,vehicle_count,vehicle_avg_age,driver_count,driver_avg_age,...,num_heavy_vehicles,late_payment_count,claim_count,total_loss_amount,first_claim_reported_date,claim_paid_amount_current_period,claim_status_current_period,days_to_first_claim_report,target,policy_length_days
906,POL-001535,INS-00309,2022-06-13,2022-01-09,2021-12-10,apd,8,8.2,4,33.0,...,2,0.0,0,0.00,NaT,0.00,No Claim,NaN,0,-30
1180,POL-000182,INS-00882,2020-06-15,2020-02-14,2020-01-15,apd,10,4.2,5,55.2,...,1,0.0,0,0.00,NaT,0.00,No Claim,NaN,0,-30
1783,POL-000294,INS-03322,2020-12-29,2020-11-04,2020-10-05,apd,5,0.0,12,42.5,...,1,0.0,0,0.00,NaT,0.00,No Claim,NaN,0,-30
2959,POL-007964,INS-03704,2021-07-12,2021-04-09,2021-03-10,al,6,5.6,7,48.9,...,2,1.0,0,0.00,NaT,0.00,No Claim,NaN,0,-30
3044,POL-011651,INS-03642,2022-03-28,2021-11-10,2021-10-11,al,12,5.6,5,37.2,...,3,0.0,0,0.00,NaT,0.00,No Claim,NaN,0,-30
3763,POL-007016,INS-00084,2022-06-05,2021-12-24,2021-11-24,apd,7,7.6,4,36.6,...,1,3.0,2,2332.81,2022-07-25,2203.56,Open,50.0,1,-30
5089,POL-003622,INS-02816,2020-08-31,2020-03-17,2020-02-16,apd,7,3.7,6,38.4,...,0,2.0,0,0.00,NaT,0.00,No Claim,NaN,0,-30
6535,POL-013043,INS-03281,2021-10-21,2021-06-08,2021-05-09,al,9,6.9,6,44.5,...,3,1.0,0,0.00,NaT,0.00,No Claim,NaN,0,-30
10582,POL-013777,INS-02741,2021-08-26,2021-08-04,2021-07-05,al,7,5.2,4,48.7,...,4,1.0,0,0.00,NaT,0.00,No Claim,NaN,0,-30
12211,POL-002965,INS-01568,2022-03-14,2022-02-06,2022-01-07,apd,9,0.5,7,42.3,...,1,3.0,0,0.00,NaT,0.00,No Claim,NaN,0,-30


In [15]:
# ---- Drop ----
df = df[df["policy_length_days"] > 0].copy()

print(f'Cleaned data: {df.shape}')

Cleaned data: (14990, 33)


#### total_vehicle_count == 0 and prior_year_mileage_000 > 0
- Drop 2 records

In [16]:
# ---- Check vehicle_count == 0 & num_heavy_vehicles == 0 & prior_year_mileage_000 > 0 ----

invalid_vehicle_mask = ( 
    (df["vehicle_count"] == 0) & (df["num_heavy_vehicles"] == 0)
    ) & (
    (df["prior_year_mileage_000"] > 0) | (df["driver_count"] > 0)
)
display(df[invalid_vehicle_mask])

,policy_id,insured_id,snapshot_date,policy_effective_date,policy_expiration_date,coverage_type,vehicle_count,vehicle_avg_age,driver_count,driver_avg_age,...,num_heavy_vehicles,late_payment_count,claim_count,total_loss_amount,first_claim_reported_date,claim_paid_amount_current_period,claim_status_current_period,days_to_first_claim_report,target,policy_length_days
9052,POL-002779,INS-03605,2021-03-01,2020-12-30,2021-12-30,apd,0,5.1,9,NaN,...,0,0.0,0,0.0,NaT,0.0,No Claim,NaN,0,365
14880,POL-006375,INS-01087,2020-06-08,2020-01-02,2021-01-01,apd,0,1.2,5,39.7,...,0,1.0,0,0.0,NaT,0.0,No Claim,NaN,0,365


In [17]:
# ---- Drop 2 rows ----
df = df[~invalid_vehicle_mask].copy()

print(f'Cleaned data: {df.shape}')

Cleaned data: (14988, 33)


#### annual_premium < 0
- Drop 5 records

In [18]:
df[df["annual_premium"] < 0 ]

,policy_id,insured_id,snapshot_date,policy_effective_date,policy_expiration_date,coverage_type,vehicle_count,vehicle_avg_age,driver_count,driver_avg_age,...,num_heavy_vehicles,late_payment_count,claim_count,total_loss_amount,first_claim_reported_date,claim_paid_amount_current_period,claim_status_current_period,days_to_first_claim_report,target,policy_length_days
2571,POL-003964,INS-01029,2021-02-18,2020-09-12,2021-09-12,apd,8,9.5,6,48.0,...,1,0.0,0,0.0,NaT,0.0,No Claim,NaN,0,365
5362,POL-003525,INS-03171,2020-11-14,2020-10-30,2021-10-30,apd,6,8.3,4,32.5,...,1,0.0,0,0.0,NaT,0.0,No Claim,NaN,0,365
10139,POL-005629,INS-01603,2021-05-01,2021-01-14,2022-01-14,apd,8,7.2,5,47.2,...,3,NaN,0,0.0,NaT,0.0,No Claim,NaN,0,365
10327,POL-008752,INS-03229,2021-10-24,2021-06-29,2022-06-29,al,13,5.6,10,38.5,...,3,1.0,0,0.0,NaT,0.0,No Claim,NaN,0,365
14455,POL-013297,INS-02261,2021-08-28,2021-07-17,2022-07-17,al,15,5.4,4,45.5,...,3,0.0,0,0.0,NaT,0.0,No Claim,NaN,0,365


In [19]:
df = df[df["annual_premium"] >= 0 ].copy()
df.shape

(14983, 33)

### Add Columns
Date Columns:
1. elapsed_days = snapshot_date - policy_effective_date
2. exposure_days = elapsed_days.clip(lower=1.0) -> if 0 put to 1
3. exposure_ratio = elapsed_days / policy_length_days                       -> use
4. remaining_days = policy_expiration_date - snapshot_date

Vehicle Columns:

5. total_vehicle_count = vehicle_count + num_heavy_vehicles
6. mileage_per_vehicle = prior_year_mileage_000 / total_vehicle_count

Claim Loss:

7. loss_per_claim = if total_prior_claims > 0 then prior_loss_amount / total_prior_claims else 0.0
8. has_large_prior_loss = prior_loss_amount > 50000
9. risk_score_external_isna = df["risk_score_external"].isna().astype(int)

#### Date Columns

In [20]:
# ---- calculate elapsed days ----
df["elapsed_days"] = (df["snapshot_date"] - df["policy_effective_date"]).dt.days

# ---- calculate exposure days with floor ---- 
df["exposure_days"] = df["elapsed_days"].clip(lower=1.0)

# ---- calculate ratio between elapsed_days and policy_length_days----
df["exposure_ratio"] = df["elapsed_days"] / df["policy_length_days"].clip(
    lower=0.0, upper=1.0)

# ---- calculate remaining days ----
df["remaining_days"] = (df["policy_expiration_date"] - df["snapshot_date"]).dt.days

df[["policy_length_days", "elapsed_days", "exposure_days", "exposure_ratio", "remaining_days"]].head()

,policy_length_days,elapsed_days,exposure_days,exposure_ratio,remaining_days
0,365,59,59,59.0,306
1,365,111,111,111.0,254
2,365,177,177,177.0,188
3,365,143,143,143.0,222
4,365,109,109,109.0,256


#### Vehicle Columns

In [21]:
# ---- total_vehicle_count ----
df['total_vehicle_count'] = df['vehicle_count'] + df['num_heavy_vehicles']
df[["vehicle_count","num_heavy_vehicles", "total_vehicle_count"]].head()

,vehicle_count,num_heavy_vehicles,total_vehicle_count
0,7,0,7
1,6,2,8
2,6,0,6
3,14,3,17
4,7,0,7


In [22]:
# ---- mileage_per_vehicle ----
df['mileage_per_vehicle'] = np.where(
    df['total_vehicle_count'] > 0, 
    df['prior_year_mileage_000'] / df['total_vehicle_count'], 
    np.nan
)

df['mileage_per_vehicle'].head()

0     5.357143
1    10.762500
2    23.533333
3          NaN
4    24.657143
Name: mileage_per_vehicle, dtype: float64

#### Claim Loss

In [23]:
# ---- loss_per_claim ----
total_prior_claims = df["prior_apd_claim_count"] + df["prior_al_claim_count"]

df['loss_per_claim'] = np.where(
    total_prior_claims > 0,
    df['prior_loss_amount'] / total_prior_claims, 
    0.0
)

df['loss_per_claim'].head()

0       0.000000
1       0.000000
2    4386.116667
3            NaN
4       0.000000
Name: loss_per_claim, dtype: float64

In [24]:
# ---- has_large_prior_loss ----
df["has_large_prior_loss"] = (df["prior_loss_amount"] > 50000).astype(int)

df["has_large_prior_loss"].value_counts()

has_large_prior_loss
0    14519
1      464
Name: count, dtype: int64

In [25]:
# ---- risk_score_external_isna ----
df["risk_score_external_isna"] = (
    df["risk_score_external"].isna().astype(int)
)

df["risk_score_external_isna"].value_counts()

risk_score_external_isna
0    14236
1      747
Name: count, dtype: int64

### Save to parquet

In [26]:
df.to_parquet(f'../data/cleaned/df_{dataset_type}_cleaned.parquet')